# Modül 3 — Veri Ön İşleme

**Eğitmen: Elif Özkan**

[LinkedIn](https://www.linkedin.com/in/eliffozkan/) · [GitHub](https://github.com/eliffozkan) · [Eğitim reposu](https://github.com/eliffozkan/GDGC-AI_ML_Course-26_27)

[YouTube — Fütürist AI (kaynak)](https://www.youtube.com/@F%C3%BCt%C3%BCrist_AIntelligence)

[Google Colab’da aç](https://colab.research.google.com/github/eliffozkan/GDGC-AI_ML_Course-26_27/blob/main/Module_3_Preprocessing/Module_3_Preprocessing_Code.ipynb)

**Ders 1 (80 dk):** tablo/özet istatistik 15, dağılım/scatter 20, IQR/box/violin 20, eksik veri ve groupby 25 dk.

**Ders 2 (75 dk):** scaling 25, encoding 25, korelasyon ve temiz tablo kontrolü 15, ödev kontrolü 10 dk.

Veri OpenML’den ilk açılışta indirilir; internet gerekir. Örneklerin amacı veri temizlemeyi göstermek; model değerlendirmesinde doldurma/ölçekleme yalnızca train üzerinde öğrenilip validation/test’e uygulanmalıdır.

**Çalıştırma:** Temiz oturumda hücreleri üstten alta çalıştırın. Kod çıktılarını ilgili açıklamayla birlikte yorumlayın.

## Ders 1 — Veriyi tanıma ve temizleme

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
np.random.seed(42)
ames = fetch_openml(name="house_prices", version=1, as_frame=True)
df = ames.frame.copy()

# Tabloyu tanıma
Mean/median/min/max/std sayısal sütunları, mode en sık değeri özetler. NaN eksik/geçersiz gözlemi temsil eder. `describe` sütunların ölçeğini, `info` türlerini ve dolu gözlem sayılarını gösterir.

In [ ]:
print("Boyut:", df.shape)
print("LotFrontage özeti:", df["LotFrontage"].describe())

In [ ]:
display(df.head())

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("LotConfig mode:", df["LotConfig"].mode().iloc[0])

Standart sapma yayılımı ölçer. Z-score ortalamadan uzaklığı std biriminde verir; Modified Z-score median/MAD kullanır. Bunları ayrı bir temizleme yöntemi olarak uygulamıyoruz.

# Skewness and bin histogram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, column in zip(axes, ["LotFrontage", "YearBuilt"]):
    ax.hist(df[column].dropna(), bins=15, edgecolor="black")
    ax.axvline(df[column].mean(), color="red", linestyle="--", label="Mean")
    ax.set(title=f"{column}: skew={df[column].skew():.2f}", xlabel=column, ylabel="Sayı")
    ax.legend()
plt.tight_layout(); plt.show()

Mean çizgisinin dağılıma göre kayması çarpıklık hakkında fikir verir. Scatter’da ilişki ve uzak gözlemler incelenir; ilişki nedensellik kanıtı değildir.

# Scatter Plot

In [ ]:
x_col = "LotFrontage"; label_col = "SalePrice"
sns.scatterplot(data=df, x=x_col, y=label_col, alpha=0.5)
plt.title("Cephe uzunluğu ve satış fiyatı"); plt.show()

# Box Plot - Violin Plot

In [ ]:
data = df['LotFrontage']
plt.figure(figsize=(8, 6))
sns.boxplot(data=data)
plt.title('Boxplot')
plt.xlabel('Değerler')
plt.show()

In [ ]:
data = df['LotFrontage']
plt.figure(figsize=(8, 6))
sns.violinplot(data=data, inner="point", color="lightblue")
plt.title("Violin Plot with Outliers")
plt.xlabel("Değerler")
plt.show()

# Handling Outliers

In [ ]:
column = "LotFrontage"

In [ ]:
Q1 = df[column].quantile(0.25)
Q3 = df[column].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR; upper = Q3 + 1.5 * IQR
outlier_mask = df[column].notna() & ~df[column].between(lower, upper)
df_outlier_deleted = df.loc[~outlier_mask].copy()
print("Aykırı satır:", outlier_mask.sum(), "Korunan eksik:", df_outlier_deleted[column].isna().sum())
sns.boxplot(x=df_outlier_deleted[column]); plt.title("Satır silme alternatifi"); plt.show()

In [ ]:
df_outlier_winsorized = df.copy()
df_outlier_winsorized[column] = df_outlier_winsorized[column].clip(lower, upper)
print("Satır sayısı korunuyor:", len(df_outlier_winsorized))
sns.boxplot(x=df_outlier_winsorized[column]); plt.title("Sınıra çekme alternatifi"); plt.show()
df = df_outlier_winsorized.copy()  # Atölyede bu alternatifle devam ediyoruz.

Satır silme ve sınıra çekme iki alternatiftir; ikisini art arda uygulamıyoruz. Eksik değerler IQR filtresinde korunur ve sonraki bölümde doldurulur.

# Missing Values

In [ ]:
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100
dtypes = df.dtypes
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage (%)': missing_percent,
    'Dtype': dtypes
})
missing_df = missing_df[missing_df['Missing Count'] > 0]
missing_df = missing_df.sort_values(by='Missing Percentage (%)', ascending=False)
missing_df

In [ ]:
threshold = df.shape[1] * 0.1
print("%10’dan fazla alanı eksik satır:", (df.isna().sum(axis=1) > threshold).sum())

In [ ]:
drop_columns = missing_df[missing_df["Missing Percentage (%)"] > 40].index.to_list()
df = df.drop(columns=drop_columns)
print("Çıkarılan yüksek eksikli sütunlar:", drop_columns)

In [ ]:
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())

In [ ]:
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['GarageYrBlt'].mean())
#df['MasVnrArea'] = df['MasVnrArea'].fillna(0)

In [ ]:
df["GarageCond"] = df["GarageCond"].astype(object).fillna("Unknown")

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category", "string"]).columns
for col in cat_cols:
    if df[col].isna().any():
        if isinstance(df[col].dtype, pd.CategoricalDtype):
            df[col] = df[col].astype(object)
        df[col] = df[col].fillna(df[col].mode(dropna=True).iloc[0])
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
print("Kalan eksik hücre:", df.isna().sum().sum())

In [ ]:
df_group = pd.DataFrame({
    'meslek': ['Mühendis', 'Doktor', 'Mühendis', 'Öğretmen', 'Doktor', 'Öğretmen'],
    'maaş': [np.nan,10000, 11000, 12000,np.nan, np.nan]
})
print(df_group)
df_group['maaş'] = df_group.groupby('meslek')['maaş'].transform(
    lambda x: x.fillna(x.mean())
)
print("*"*25)
print(df_group)

Mean uç değerlere daha duyarlı, median daha dayanıklıdır; kategorik sütunda mode kullanılabilir. GroupBy aynı gruptaki gözlemlerin ortalamasını kullanır.

---
## Ders 2 — Ölçekleme, encoding ve sonuç kontrolü
Ders 1’de outlier sınıra çekme ve eksik doldurma sonrası oluşturduğumuz df üzerinden devam ediyoruz.

# Feature Scaling

In [ ]:
scaler = MinMaxScaler()
df['LotFrontage_scaled'] = scaler.fit_transform(df[['LotFrontage']])

df['LotFrontage'].plot(kind='line', figsize=(8, 5))
plt.show()
df['LotFrontage_scaled'].plot(kind='line', figsize=(8, 5))
plt.show()

In [ ]:
data = np.random.normal(loc=50, scale=10, size=(1000, 1))
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

plt.figure(figsize=(12,5))
plt.subplot(1, 2, 1)
plt.hist(data, bins=30, density=True, alpha=0.6, color='blue')
plt.title('Orijinal Veri')
plt.xlabel('Değer')
plt.ylabel('Yoğunluk')

plt.subplot(1, 2, 2)
plt.hist(data_scaled, bins=30, density=True, alpha=0.6, color='green')
plt.title('StandardScaler ile Ölçeklenmiş Veri')
plt.xlabel('Değer')
plt.ylabel('Yoğunluk')

plt.tight_layout()
plt.show()

print(f"Orijinal veri ortalaması: {np.mean(data):.4f}, std: {np.std(data):.4f}")
print(f"Ölçeklenmiş veri ortalaması: {np.mean(data_scaled):.4f}, std: {np.std(data_scaled):.4f}")


MinMax eğitimdeki değerleri 0–1 aralığına taşır; StandardScaler mean=0/std=1 yapar, dağılımı normal dağılıma dönüştürmez. İkisi de outlier’dan etkilenebilir. MaxAbs ve Robust başka scaling seçenekleridir; bu dönem yalnızca isimlerini bilmek yeterlidir.

# Finding Unique Values

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category", "string"]).columns
for col in ["LotConfig", "LandContour"]:
    print(col, df[col].unique())

# Encoding

In [ ]:
le = LabelEncoder()
print(df["LotConfig"].unique())
df['LotConfig'] = le.fit_transform(df['LotConfig'])
print(df["LotConfig"].unique())

Label encoding kategoriye sayı atar; bu sayılar LotConfig için gerçek bir sıralama değildir. Lineer modellerde sahte sıralamadan kaçınmak için one-hot tercih edilir. Ordinal anlamlı sıralama, Frequency kategori sıklığı kullanır; burada uygulamıyoruz.

In [ ]:
df_encoded = pd.get_dummies(df, columns=["LandContour"], dtype=int)
print("Yeni one-hot sütunları:", [c for c in df_encoded.columns if c.startswith("LandContour_")])
display(df_encoded.filter(like="LandContour_").head())

In [ ]:
print("Temiz tablo boyutu:", df_encoded.shape)
print("Kalan eksik hücre:", df_encoded.isna().sum().sum())

# Feature Selection

In [ ]:
selected_columns = ["LotFrontage", "GrLivArea", "YearBuilt", "SalePrice"]
corr_matrix = df_encoded[selected_columns].corr()
plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Seçili özelliklerin Pearson korelasyonu"); plt.tight_layout(); plt.show()

Korelasyon -1/+1 yönlü lineer ilişkiyi, 0 zayıf lineer ilişkiyi gösterir; doğrusal olmayan ilişkiyi dışlamaz. Mutual Information başka bir ilişki ölçüsüdür; bu dönem hesaplamıyoruz. `df_encoded` eksiksiz fakat diğer kategorik sütunlar hâlâ metin olabilir; tüm sütunlarla model eğitimi için onların da uygun encoding’i gerekir. Ödev yalnızca belirtilen eksikleri doldurma ve ID sütununu çıkarma üzerinedir.

**Modül materyalleri:** [Module_3_Preprocessing](https://github.com/eliffozkan/GDGC-AI_ML_Course-26_27/tree/main/Module_3_Preprocessing)

Ders kodu ve ödev dosyaları bu modül klasöründedir.